In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("HW_3b_robot_arm_reach.ipynb")

# Robot arm - reach for a goal

Use **fmin** to find joint angles that get the end location to a target. 

Slides: https://docs.google.com/presentation/d/17aiTBmPZidR6op7TvqYRzYatuc_NETYA1BhgpSHQ-FM/edit?usp=sharing

In [ ]:
# The usual imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fmin

In [ ]:
# matrix_rouintes.py functions
import matrix_routines as mt

# These are the class you wrote in the previous lab/first part of the homework
from arm_component import ArmComponent
from robot_arm import RobotArm2D

In [ ]:
# These commands will force JN to actually re-load the external file when you re-execute the import command
%load_ext autoreload
%autoreload 2

# FMin optimization

In this problem you're going to write a function **distance_from_angles_for_fmin** that you will pass to fmin in order to have **fmin** find the angles that bring the end location of the last link to the target points

GUIDES: Edit **do_fmin** to call **func_for_fmin** with the appropriate angles, etc


In [ ]:
def distance_from_angles_for_fmin(angles, robot_arm:RobotArm2D, target):
    """ Compute the distance from the grasp point to the target
    @param angles as a numpy array, one angle for each joint
    @param robot_arm - an instance of RobotArm2D
    @param target - a tuple with the desired x,y location
    @return The distance between the target and the gripper grasp point"""

    # GUIDES: step 1, convert the numpy array into the format we've been using for the angles
    #  [a1, a2, a3, ... ]
    # step 2, call the set_link_angles method on robot_arm with the angle list
    # step 3, get the gripper location
    # step 4: calculate the distance between the target and the gripper

    ...

    # GUIDE: Call set_link_angles on the instance of robot_arm with the angles you just created
    #    This sets the matrix_pose's to the correct values for these angles
    ...

    # GUIDE: Get the end location and calculate the distance 

    ...
    ... # Return distance


In [ ]:
# Make an instance of the arm
base_size_param = (0.5, 1.0)   # Height, width
link_sizes_param = [(0.5, 0.25), (0.3, 0.1), (0.2, 0.05)]

robot_arm = RobotArm2D(base_arm_size=base_size_param, link_sizes=link_sizes_param)

# Set the starting angles of the arm
angles_start = [np.pi/6.0, -np.pi/4, 1.5 * np.pi/4]
robot_arm.set_link_angles(angles_start)

# Get the end location
end_loc = robot_arm.get_end_location()

# Check that your function returns zero
#   This converts the start angles to a numpy array
angles_start_np = np.array(angles_start)
dist = distance_from_angles_for_fmin(angles=angles_start_np, 
                                     robot_arm=robot_arm, 
                                     target=(end_loc[0], end_loc[1]))

assert np.isclose(dist, 0.0)

# Try some angles that should be further away
angles_not_start_as_np_array = angles_start_np * 0.5
dist_far_away = distance_from_angles_for_fmin(angles=angles_not_start_as_np_array, 
                                     robot_arm=robot_arm, 
                                     target=(end_loc[0], end_loc[1]))

assert dist_far_away > 0.0

In [ ]:
grader.check("optimization_dist_func")

## Do the fmin call

Here you're going to set up the **fmin** call. As always, you can write this outside of the function call then put it in the function after it's working

- First, convert the starting angles to a numpy array (see previous question)
- Next, call fmin with the function distance_from_angles_for_fmin and a tuple that has an instance of robot arm and the target in it


In [ ]:
def do_fmin(angles_start, robot_arm:RobotArm2D, target):
    """ Set the angles/matrices of arm_geometry so they reach the target point
    @param - angles to start with
    @param - robot_arm - an instance of a robot arm
    @param - the target as a tuple (x,y)
    @return the robot arm with the angles set to the best ones
    """
    # GUIDES: 1: convert the list of angles to a numpy array (see above)
    # GUIDES: 2: call fmin with the distance_from_angles_for_fmin function
    # GUIDES: 3: set the angles of the arm to the angles returned from distance_from_angles_for_fmin

    ...
    return robot_arm

In [ ]:
# You must start with these angles
angles_start = [np.pi/6.0, -np.pi/4, 1.5 * np.pi/4]
# The target
target = np.array([0.55, 1.15])

arm_geometry_optimized = do_fmin(angles_start, robot_arm, target)

In [ ]:
assert np.isclose(robot_arm.get_end_location()[0], target[0], atol=0.01)
assert np.isclose(robot_arm.get_end_location()[1], target[1], atol=0.01)

In [ ]:
# Plot arm with target
fig, axs = plt.subplots(1, 1, figsize=(6, 6))
robot_arm.plot(axs=axs)
axs.plot(target[0], target[1], '+r', markersize=20)

In [ ]:
grader.check("do_fmin")

# Generalization

If nothing has been "hardwired" in, this should just work - changing the geometry, the starting angles, the target point. However, if you've hardwired something in, it probably won't...

In [ ]:
# Create the arm geometry
base_size_param = (0.5, 0.25) # squished
link_sizes_param = [(0.3, 0.15), (0.2, 0.09), (0.1, 0.05), (0.075, 0.03)]

# Create the new arm
robot_arm_longer = RobotArm2D(base_arm_size=base_size_param, link_sizes=link_sizes_param)

# Set the angles of the arm
angles_start_longer = [-np.pi/4.0, -np.pi/4, 1.2 * np.pi/4, -1 * np.pi/8]
robot_arm_longer.set_link_angles(angles_start_longer)

target_longer = np.array([0.3, 0.15])

In [ ]:
# Plot arm with target
fig, axs = plt.subplots(1, 1, figsize=(8, 8))
robot_arm_longer.plot(axs)
axs.plot(target_longer[0], target_longer[1], '+r', markersize=20, label="target")

In [ ]:
# Do the optimization
arm_longer_optimized = do_fmin(angles_start_longer, robot_arm_longer, target_longer)

In [ ]:
# Plot arm with target
fig, axs = plt.subplots(1, 1, figsize=(8, 8))
arm_longer_optimized.plot(axs)
axs.plot(target_longer[0], target_longer[1], '+r', markersize=20, label="target")

In [ ]:
assert np.isclose(arm_longer_optimized.get_end_location()[0], target_longer[0], atol=0.01)
assert np.isclose(arm_longer_optimized.get_end_location()[1], target_longer[1], atol=0.01)


In [ ]:
grader.check("generalization")

## Hours and collaborators
Required for every assignment - fill out before you hand-in.

Listing names and websites helps you to document who you worked with and what internet help you received in the case of any plagiarism issues. You should list names of anyone (in class or not) who has substantially helped you with an assignment - or anyone you have *helped*. You do not need to list TAs.

Listing hours helps us track if the assignments are too long.

In [ ]:
import os

# List of names (creates a set)
worked_with_names = {"not filled out"}
# List of URLS FA26 (creates a set)
websites = {"not filled out"}
# Approximate number of hours, including lab/in-class time
hours = -1.5

# VS Code stores the path in a special global variable
if '__vsc_ipynb_file__' in globals():
    notebook_path = globals()['__vsc_ipynb_file__']
    notebook_name = os.path.basename(notebook_path)
    json_name = notebook_name[:-6] + "_source.json"
    if not os.path.exists(json_name):
        print(f"Could not find the json file {json_name}; make sure the required VSCode extension is installed and running")


In [ ]:
grader.check("hours_collaborators")

### To submit

(Did you read me?)

- Submit this .ipynb file, the .json file.
- If you are using your own .py files, make sure to include arm_component.py AND robot_arm.py. If you don't include the.py Gradescope cannot magically reach out to your computer and find it.
- If you are using our .py files, do not include your .py files
- We will supply matrix_routines.py for you (it won't break anything if you do include it).
- As always, do a restart-runall before submitting and make sure your plots are visible.

If the Gradescope autograder fails, please check here first for common reasons for it to fail
    https://docs.google.com/presentation/d/1tYa5oycUiG4YhXUq5vHvPOpWJ4k_xUPp2rUNIL7Q9RI/edit?usp=sharing

Lots of people forget arm_component.py. Please check your autograder score to see if you are one of them.

Make sure you remove all the print statements you put in that print out lots of stuff.